# 02 — Quality control, metadata and 5-fold splits

Starts from the preprocessed volumes (`z × 380 × 380`, intensities in [0, 1]).

1. Health check of all 172 volumes (NaN/Inf, intensity range, shape).
2. Join with the clinical label table → `dataset_metadata.csv` (EGFR Wildtype = 0, Mutant = 1: 129 vs 43 patients), with a patient-level **stratified 5-fold** split (`random_state=42`).
3. A PyTorch `Dataset` / `DataLoader` smoke test for the 2D slice input.

Paths are Windows paths from the original machine (`D:\\NSCLC_PROJECT\\...`); change the path constants at the top of each notebook before running. Raw data and model weights are **not** included in this repository.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = r"D:\NSCLC_PROJECT\preprocessed_data"
QC_DIR = os.path.join(OUTPUT_DIR, "qc_plots")

# ۱. بررسی تعداد دقیق فایل‌ها
npy_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('_preprocessed.npy')]
qc_files = [f for f in os.listdir(QC_DIR) if f.endswith('_QC.png')]

print(f"📊 Total preprocessed .npy files: {len(npy_files)} / 172")
print(f"🖼️ Total QC plot images: {len(qc_files)} / 172")

# ۲. تست سلامت محتوا و ابعاد (دیتاهای پوچ یا NaN)
corrupted_files = []
shapes = []
stats = []

for f in npy_files:
    path = os.path.join(OUTPUT_DIR, f)
    try:
        data = np.load(path)
        # بررسی وجود مقادیر NaN یا Inf
        if np.isnan(data).any() or np.isinf(data).any():
            corrupted_files.append((f, "NaN/Inf values found"))
        # بررسی بازه نرمال‌سازی [0, 1]
        elif data.min() < 0.0 or data.max() > 1.0:
            corrupted_files.append((f, f"Out of range: [{data.min()}, {data.max()}]"))
        else:
            shapes.append(data.shape)
            stats.append({'min': data.min(), 'max': data.max(), 'mean': data.mean()})
    except Exception as e:
        corrupted_files.append((f, str(e)))

print("\n--- 🔍 Health Check Results ---")
if not corrupted_files:
    print("✅ All 172 files are completely HEALTHY! No NaN, Inf, or invalid ranges.")
else:
    print(f"⚠️ Found {len(corrupted_files)} corrupted files:")
    for file_name, reason in corrupted_files:
        print(f"   - {file_name}: {reason}")

# ۳. نمایش میانگین ابعاد مکانی
z_sizes = [s[0] for s in shapes]
print(f"\n📏 Spatial Dimensions (XY): Always {shapes[0][1]}x{shapes[0][2]}")
print(f"📐 Z-Slice Range: Min={min(z_sizes)}, Max={max(z_sizes)}, Mean={np.mean(z_sizes):.1f} slices")

📊 Total preprocessed .npy files: 172 / 172
🖼️ Total QC plot images: 172 / 172

--- 🔍 Health Check Results ---
✅ All 172 files are completely HEALTHY! No NaN, Inf, or invalid ranges.

📏 Spatial Dimensions (XY): Always 380x380
📐 Z-Slice Range: Min=146, Max=1684, Mean=452.0 slices


In [4]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# ۱. تنظیم دقیق مسیرها
OUTPUT_DIR = r"D:\NSCLC_PROJECT\preprocessed_data"
LABEL_CSV_PATH = r"D:\NSCLC_PROJECT\raw-zip\labels\NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv"
SAVE_METADATA_PATH = r"D:\NSCLC_PROJECT\dataset_metadata.csv"

# ۲. استخراج شناسه ۱۷۲ بیمار پیش‌پردازش‌شده
processed_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('_preprocessed.npy')]
valid_patient_ids = {f.split('_preprocessed')[0] for f in processed_files}

print(f"📁 تعداد فایل‌های npy پیدا شده: {len(valid_patient_ids)}")

# ۳. بارگذاری جدول لیبل‌ها و تمیزسازی اولیه
df_labels = pd.read_csv(LABEL_CSV_PATH)
df_labels.columns = df_labels.columns.str.strip()
df_labels['clean_id'] = df_labels['Case ID'].astype(str).str.strip()

# ۴. فیلتر کردن بیمارانی که پیش‌پردازش موفق داشته‌اند
df_final = df_labels[df_labels['clean_id'].isin(valid_patient_ids)].copy()

# ۵. نگاشت وضعیت جهش EGFR به مقادیر عددی (Wildtype = 0, Mutant = 1)
df_final['label'] = df_final['EGFR mutation status'].map({'Wildtype': 0, 'Mutant': 1})
df_final['file_path'] = df_final['clean_id'].apply(lambda pid: os.path.join(OUTPUT_DIR, f"{pid}_preprocessed.npy"))

# اطمینان از عدم وجود لیبل خالی
df_final = df_final.dropna(subset=['label']).reset_index(drop=True)
df_final['label'] = df_final['label'].astype(int)

# ۶. تقسیم‌بندی Stratified 5-Fold با حفظ نسبت کلاس‌ها در سطح بیمار
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
df_final['fold'] = -1

for fold, (train_idx, val_idx) in enumerate(skf.split(df_final, df_final['label'])):
    df_final.loc[val_idx, 'fold'] = fold

# ۷. ذخیره فایل متادیتا جهت استفاده در پایتورچ (PyTorch)
df_final[['clean_id', 'label', 'fold', 'file_path']].to_csv(SAVE_METADATA_PATH, index=False)

print("\n--------------------------------------------------")
print(f"✅ فایل متادیتا با موفقیت در مسیر زیر ذخیره شد:\n{SAVE_METADATA_PATH}")
print("--------------------------------------------------")
print("\n📊 توزیع نهایی کلاس‌ها در دیتاسِت:")
print(df_final['label'].value_counts().rename({0: 'Wildtype (Class 0)', 1: 'Mutant (Class 1)'}))

print("\n📋 توزیع کلاس‌ها در هر Fold جهت اطمینان از بالانس بودن:")
print(pd.crosstab(df_final['fold'], df_final['label'], margins=True))

📁 تعداد فایل‌های npy پیدا شده: 172

--------------------------------------------------
✅ فایل متادیتا با موفقیت در مسیر زیر ذخیره شد:
D:\NSCLC_PROJECT\dataset_metadata.csv
--------------------------------------------------

📊 توزیع نهایی کلاس‌ها در دیتاسِت:
label
Wildtype (Class 0)    129
Mutant (Class 1)       43
Name: count, dtype: int64

📋 توزیع کلاس‌ها در هر Fold جهت اطمینان از بالانس بودن:
label    0   1  All
fold               
0       26   9   35
1       26   9   35
2       26   8   34
3       26   8   34
4       25   9   34
All    129  43  172


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

class NSCLCDataset(Dataset):
    def __init__(self, metadata_path, fold=0, is_train=True, transform=None):
        df = pd.read_csv(metadata_path)
        # تفکیک بر اساس Fold
        if is_train:
            self.df = df[df['fold'] != fold].reset_index(drop=True)
        else:
            self.df = df[df['fold'] == fold].reset_index(drop=True)
            
        self.is_train = is_train
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['file_path']
        label = row['label']

        # بارگذاری حجم ۳بعدی (Z, H, W)
        volume = np.load(file_path)

        # استخراج اسلایس کلیدی (اسلایسی که بیشترین بافت ریه/تومور را دارد)
        slice_counts = [(slice_arr > 0.05).sum() for slice_arr in volume]
        best_slice_idx = np.argmax(slice_counts)
        img_slice = volume[best_slice_idx] # ابعاد: (380, 380)

        # تبدیل به ۳ کانال (RGB) برای شبکه‌های پیش‌آموخته ViT
        tensor_img = torch.tensor(img_slice, dtype=torch.float32).unsqueeze(0)
        tensor_img = torch.cat([tensor_img] * 3, dim=0) # ابعاد: (3, 380, 380)

        # ریسایز ساده به 224x224 (ابعاد استاندارد ورودی ViT)
        tensor_img = torch.nn.functional.interpolate(
            tensor_img.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False
        ).squeeze(0)

        # Data Augmentation ساده در زمان آموزش
        if self.is_train:
            if torch.rand(1).item() > 0.5:
                tensor_img = torch.flip(tensor_img, dims=[2]) # Flip افقی

        return tensor_img, torch.tensor(label, dtype=torch.long)

# تست پایداری لودر داده روی Fold 0
metadata_path = r"D:\NSCLC_PROJECT\dataset_metadata.csv"
train_dataset = NSCLCDataset(metadata_path, fold=0, is_train=True)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# دریافت یک بچ نمونه جهت تست
images, labels = next(iter(train_loader))
print(f"✅ DataLoader test successful!")
print(f"📦 Batch Images Shape: {images.shape} (Batch, Channels, Height, Width)")
print(f"🏷️ Batch Labels: {labels}")

✅ DataLoader test successful!
📦 Batch Images Shape: torch.Size([4, 3, 224, 224]) (Batch, Channels, Height, Width)
🏷️ Batch Labels: tensor([0, 0, 1, 0])
